<a href="https://colab.research.google.com/github/thalbl/real-time-translator/blob/main/ml_final_project_t.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Real-Time Translator with AI

Academic project: **STT (Whisper) -> Translation (M2M100) -> TTS (SpeechT5)**
Adapted for Google Colab with browser recording and interactive interface.

| Stage | Model | Purpose |
|-------|-------|---------|
| STT | OpenAI Whisper | Speech -> Text |
| Translation | Meta M2M100 | Text language A -> Text language B |
| TTS | Microsoft SpeechT5 | Text -> Synthesized speech |

In [1]:
#C2
!pip install -q torch transformers sentencepiece scipy numpy

import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    print(f"GPU detected {torch.cuda.get_device_name(0)}")
    print(f" VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("No GPU! Runtime → Change runtime type → T4 GPU")


No GPU! Runtime → Change runtime type → T4 GPU


In [2]:
# C3 - Clean up old models from memory
import gc
try: del stt
except: pass
try: del translator
except: pass
try: del tts
except: pass
gc.collect()
torch.cuda.empty_cache()
print(f"Memory cleared! Free VRAM: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

AssertionError: Torch not compiled with CUDA enabled

In [ ]:
# ============================================================
# C4 - Imports
# ============================================================
import warnings
warnings.filterwarnings("ignore")

import os, base64, tempfile, subprocess
import numpy as np
import torch
from IPython.display import display, Audio as IPAudio, Javascript, HTML, clear_output
import ipywidgets as widgets
import scipy.io.wavfile as wavfile

from transformers import (
    pipeline,
    M2M100ForConditionalGeneration,
    M2M100Tokenizer,
    SpeechT5ForTextToSpeech,
    SpeechT5HifiGan,
    SpeechT5Processor,
)

from google.colab import output as colab_output

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE.upper()}")

In [ ]:
# ============================================================
# Cell 5 - Browser Audio Recording (replaces sounddevice for Colab)
# ============================================================

RECORD_JS = """
async function recordAudio(seconds) {
    const stream = await navigator.mediaDevices.getUserMedia({audio: true});
    const recorder = new MediaRecorder(stream, {mimeType: 'audio/webm;codecs=opus'});
    const chunks = [];

    recorder.ondataavailable = e => chunks.push(e.data);
    recorder.start();

    // Visual recording indicator
    const el = document.createElement('div');
    el.style.cssText = `
        padding: 12px 24px; margin: 8px 0;
        background: linear-gradient(135deg, #e74c3c, #c0392b);
        color: #fff; border-radius: 10px; font-size: 15px;
        text-align: center; font-weight: 600;
        box-shadow: 0 4px 15px rgba(231,76,60,0.4);
        animation: recPulse 1.5s ease-in-out infinite;
    `;
    el.textContent = 'Recording... speak now! (' + seconds + 's)';
    const style = document.createElement('style');
    style.textContent = '@keyframes recPulse{0%,100%{transform:scale(1);opacity:1}50%{transform:scale(1.02);opacity:0.7}}';
    document.head.appendChild(style);
    document.querySelector('#output-area').appendChild(el);

    await new Promise(r => setTimeout(r, seconds * 1000));

    const stopped = new Promise(r => recorder.onstop = r);
    recorder.stop();
    await stopped;
    stream.getTracks().forEach(t => t.stop());
    el.remove(); style.remove();

    const blob = new Blob(chunks, {type: 'audio/webm'});
    const reader = new FileReader();
    return await new Promise(r => {
        reader.onloadend = () => r(reader.result.split(',')[1]);
        reader.readAsDataURL(blob);
    });
}
"""


def record_browser_audio(duration_seconds=5):
    """
    Record audio from the browser microphone.
    Returns: np.ndarray (mono, 16kHz, float32)
    """
    print(f"Preparing {duration_seconds}s recording...")
    display(Javascript(RECORD_JS))

    # Capture audio via JavaScript -> base64 (WebM)
    b64_audio = colab_output.eval_js(f'recordAudio({duration_seconds})')
    audio_bytes = base64.b64decode(b64_audio)

    # Save as WebM and convert to WAV 16kHz mono with ffmpeg
    tmp_webm = tempfile.mktemp(suffix='.webm')
    tmp_wav  = tempfile.mktemp(suffix='.wav')

    with open(tmp_webm, 'wb') as f:
        f.write(audio_bytes)

    subprocess.run(
        ['ffmpeg', '-y', '-i', tmp_webm, '-ar', '16000', '-ac', '1', '-f', 'wav', tmp_wav],
        capture_output=True
    )

    sr, audio = wavfile.read(tmp_wav)
    audio = audio.astype(np.float32) / 32768.0  # int16 -> float32

    os.remove(tmp_webm)
    os.remove(tmp_wav)

    print(f"Audio captured: {len(audio)} samples ({len(audio)/16000:.1f}s)")
    return audio


print("record_browser_audio() function ready!")

In [ ]:
# ============================================================
# Cell 6 - STT Module (Speech-to-Text) -- Whisper
# ============================================================
from transformers import WhisperProcessor, WhisperForConditionalGeneration

class STTModule:
    """Converts speech (audio) to text using Whisper."""

    def __init__(self, model_name="openai/whisper-tiny", device=DEVICE):
        print(f"[STT] Loading model '{model_name}' on '{device}'...")
        self.processor = WhisperProcessor.from_pretrained(model_name)
        self.model = WhisperForConditionalGeneration.from_pretrained(model_name).to(device)
        self.device = device
        self.sample_rate = 16000
        print("[STT] Model loaded!")

    def transcribe(self, audio):
        """Transcribe a numpy audio array to text."""
        input_features = self.processor(
            audio,
            sampling_rate=self.sample_rate,
            return_tensors="pt",
        ).input_features.to(self.device)

        predicted_ids = self.model.generate(
            input_features,
            language="portuguese",
        )

        text = self.processor.batch_decode(
            predicted_ids, skip_special_tokens=True
        )[0].strip()
        print(f"[STT] Recognized text: '{text}'")
        return text

    def record_and_transcribe(self, duration_seconds=5.0):
        """Record from the browser microphone and transcribe."""
        audio = record_browser_audio(duration_seconds)
        return self.transcribe(audio)


print("STTModule class defined!")

In [ ]:
# ============================================================
# C7 - Translation Module -- M2M100 (Multilingual)
# ============================================================

class TranslatorModule:
    """Translates text between languages using M2M100."""

    SUPPORTED_LANGUAGES = {
        "pt": "Portuguese", "en": "English", "es": "Spanish",
        "no": "Norwegian", "fr": "French", "de": "German",
        "it": "Italian", "zh": "Chinese", "ja": "Japanese",
        "ko": "Korean", "ru": "Russian", "ar": "Arabic",
    }

    def __init__(self, source_lang="pt", target_lang="en",
                 model_name="facebook/m2m100_418M", device=DEVICE):
        self.source_lang = source_lang
        self.target_lang = target_lang
        self.device = device

        src_name = self.SUPPORTED_LANGUAGES.get(source_lang, source_lang)
        tgt_name = self.SUPPORTED_LANGUAGES.get(target_lang, target_lang)

        print(f"[Translator] Loading '{model_name}'...")
        print(f"[Translator] Pair: {src_name} -> {tgt_name}")
        self.tokenizer = M2M100Tokenizer.from_pretrained(model_name)
        self.model = M2M100ForConditionalGeneration.from_pretrained(model_name).to(device)
        self.tokenizer.src_lang = source_lang
        print("[Translator] Model loaded!")

    def translate(self, text):
        """Translate text to the target language."""
        if not text or not text.strip():
            return ""

        self.tokenizer.src_lang = self.source_lang
        encoded = self.tokenizer(text, return_tensors="pt").to(self.device)

        generated = self.model.generate(
            **encoded,
            forced_bos_token_id=self.tokenizer.get_lang_id(self.target_lang),
        )

        result = self.tokenizer.batch_decode(
            generated, skip_special_tokens=True
        )[0].strip()

        print(f"[Translator] '{text}' -> '{result}'")
        return result

    def switch_languages(self, source_lang, target_lang):
        """Switch language pair without reloading the model."""
        self.source_lang = source_lang
        self.target_lang = target_lang
        self.tokenizer.src_lang = source_lang
        src_name = self.SUPPORTED_LANGUAGES.get(source_lang, source_lang)
        tgt_name = self.SUPPORTED_LANGUAGES.get(target_lang, target_lang)
        print(f"[Translator] Pair changed: {src_name} -> {tgt_name}")


print("TranslatorModule class defined!")

In [ ]:
# ============================================================
# C8 - TTS Module (Text-to-Speech) -- SpeechT5
# ============================================================

class TTSModule:
    """Converts text to speech using SpeechT5."""

    def __init__(self, device=DEVICE):
        model_name = "microsoft/speecht5_tts"
        vocoder_name = "microsoft/speecht5_hifigan"

        print(f"[TTS] Loading model '{model_name}'...")
        self.processor = SpeechT5Processor.from_pretrained(model_name)
        self.model = SpeechT5ForTextToSpeech.from_pretrained(model_name).to(device)
        self.vocoder = SpeechT5HifiGan.from_pretrained(vocoder_name).to(device)
        self.speaker_embedding = torch.zeros(1, 512).to(device)
        self.device = device
        self.sample_rate = 16000
        print("[TTS] Model loaded!")

    def synthesize(self, text):
        """Convert text to a NumPy audio array."""
        if not text or not text.strip():
            return np.array([], dtype=np.float32)

        inputs = self.processor(text=text, return_tensors="pt").to(self.device)

        with torch.no_grad():
            speech = self.model.generate_speech(
                inputs["input_ids"],
                self.speaker_embedding,
                vocoder=self.vocoder,
            )

        audio = speech.cpu().numpy()
        print(f"[TTS] Audio generated: '{text[:50]}...' ({len(audio)} samples)")
        return audio

    def speak(self, text):
        """Convert text to speech and play it in the notebook."""
        audio = self.synthesize(text)
        if len(audio) == 0:
            print("[TTS] Empty text.")
            return

        print("[TTS] Playing audio...")
        display(IPAudio(audio, rate=self.sample_rate, autoplay=True))


print("TTSModule class defined!")

In [ ]:
# ============================================================
# C9 - Load all AI models
# ============================================================
print("Loading models... this may take a few minutes.\n")

# Re-initializing models to ensure they use the latest class definitions.
stt = STTModule(model_name="openai/whisper-medium", device=DEVICE)
print()

translator = TranslatorModule(source_lang="pt", target_lang="en", model_name="facebook/m2m100_1.2B", device=DEVICE)

print()

tts = TTSModule(device=DEVICE)

print("\n" + "=" * 50)
print("ALL MODELS LOADED SUCCESSFULLY!")
print("=" * 50)

In [ ]:
# ============================================================
# C11 - STT Test
# ============================================================

# Test: Speech-to-Text
print("Recording 5 seconds...")
audio = record_browser_audio(5)
text = stt.transcribe(audio)
print(f"\nResult: {text}")

# Play back the captured audio:
display(IPAudio(audio, rate=16000))

In [ ]:
# C3 - Test: Text-to-Speech
tts.speak("Hello, how are you today? This is a test.")

In [ ]:
# ============================================================
# C15 - API version dependencies
# ============================================================
!pip install -q SpeechRecognition deep-translator gTTS pydub

print("API version dependencies installed!")

In [ ]:
# ============================================================
# C18 - TTS via API -- Google Text-to-Speech (gTTS)
# ============================================================
from gtts import gTTS
import tempfile

class TTSModuleAPI:
    """Converts text to speech using Google TTS free API."""

    # Language code mapping for gTTS
    LANG_MAP = {
        "pt": "pt", "en": "en", "es": "es", "no": "no",
        "fr": "fr", "de": "de", "it": "it", "ja": "ja",
        "ko": "ko", "ru": "ru", "ar": "ar", "zh-CN": "zh-CN",
    }

    def __init__(self):
        print("[TTS-API] Google TTS ready!")

    def speak(self, text, language="en"):
        """Convert text to speech and play it in the notebook."""
        if not text or not text.strip():
            print("[TTS-API] Empty text.")
            return

        lang = self.LANG_MAP.get(language, language)
        audio_tts = gTTS(text=text, lang=lang, slow=False)

        # Save as temporary MP3 and play
        tmp = tempfile.mktemp(suffix='.mp3')
        audio_tts.save(tmp)

        print(f"[TTS-API] Playing: '{text[:50]}...'")
        display(IPAudio(tmp, autoplay=True))

        # Clean up after a delay
        import threading
        threading.Timer(10, lambda: os.remove(tmp) if os.path.exists(tmp) else None).start()


tts_api = TTSModuleAPI()

In [ ]:
# C25

!pip install -q datasets==2.20.0 transformers==4.44.0

In [ ]:
# C26

from datasets import load_dataset

fleurs_pt = load_dataset("google/fleurs", "pt_br", split="test", trust_remote_code=True)
fleurs_en = load_dataset("google/fleurs", "en_us", split="test", trust_remote_code=True)
fleurs_no = load_dataset("google/fleurs", "nb_no", split="test", trust_remote_code=True)

print(f"Samples PT: {len(fleurs_pt)}")
print(f"Samples EN: {len(fleurs_en)}")
print(f"Samples NO: {len(fleurs_no)}")

In [ ]:
# C27
print(f"\nExample keys: {list(fleurs_no[0].keys())}")
print(f"Audio shape: {fleurs_pt[0]['audio']['array'].shape}")
print(f"Transcription: {fleurs_no[0]['transcription'][:80]}...")

In [ ]:
# ============================================================
# C28 - Supervised Learning -- Language Detection Classifier
# Goal: Train a classifier that automatically detects which
# language is being spoken (PT / EN / NO) from audio features.
# This could replace manual language selection in the translator.
# ============================================================
import librosa
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns

# -- 1. Pre-processing: Extract MFCC features from audio ------
def extract_features(audio_array, sr=16000):
    """
    Extract MFCC (Mel-Frequency Cepstral Coefficients) from an audio array.
    Returns 26 features: mean + std of 13 MFCC coefficients.
    """
    mfccs = librosa.feature.mfcc(y=audio_array, sr=sr, n_mfcc=13)
    mean = mfccs.mean(axis=1)
    std = mfccs.std(axis=1)
    return np.concatenate([mean, std])  # 26 features total


# -- 2. Build the dataset -------------------------------------
print("Extracting MFCC features from audio samples...")

X, y = [], []

for i in range(min(200, len(fleurs_pt))):
    features = extract_features(fleurs_pt[i]["audio"]["array"])
    X.append(features)
    y.append("pt")

for i in range(min(200, len(fleurs_en))):
    features = extract_features(fleurs_en[i]["audio"]["array"])
    X.append(features)
    y.append("en")

for i in range(min(200, len(fleurs_no))):
    features = extract_features(fleurs_no[i]["audio"]["array"])
    X.append(features)
    y.append("no")


X = np.array(X)
y = np.array(y)
print(f"Dataset: {X.shape[0]} samples, {X.shape[1]} features, 3 languages")


# -- 3. Train/test split --------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {len(X_train)} samples | Test: {len(X_test)} samples")


# -- 4. Pre-processing: Feature normalization ------------------
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


# -- 5. Train supervised models --------------------------------

# Model 1: Logistic Regression
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_scaled, y_train)
y_pred_lr = lr.predict(X_test_scaled)

# Model 2: Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train_scaled, y_train)
y_pred_rf = rf.predict(X_test_scaled)


# -- 6. Evaluation metrics ------------------------------------
print("=" * 55)
print("LOGISTIC REGRESSION")
print("=" * 55)
print(classification_report(y_test, y_pred_lr))
print(f"Accuracy: {accuracy_score(y_test, y_pred_lr):.2%}")

print("\n" + "=" * 55)
print("RANDOM FOREST")
print("=" * 55)
print(classification_report(y_test, y_pred_rf))
print(f"Accuracy: {accuracy_score(y_test, y_pred_rf):.2%}")


# -- 7. Confusion Matrix visualization ------------------------
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, y_pred, title in [
    (axes[0], y_pred_lr, "Logistic Regression"),
    (axes[1], y_pred_rf, "Random Forest"),
]:
    cm = confusion_matrix(y_test, y_pred, labels=["pt", "en", "no"])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["PT", "EN", "NO"],
                yticklabels=["PT", "EN", "NO"], ax=ax)
    ax.set_title(f"Confusion Matrix -- {title}", fontweight="bold")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# C29 - Unsupervised Learning -- K-Means Audio Clustering
# Goal: Cluster audio samples by acoustic features WITHOUT
# using language labels, then check if the discovered clusters
# naturally correspond to the actual languages.
# ============================================================
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.decomposition import PCA

# Use the same MFCC features (X) already extracted
X_all_scaled = scaler.transform(X)

# -- K-Means Clustering (3 clusters for 3 languages) ----------
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_all_scaled)

# -- Clustering evaluation metrics ----------------------------
ari = adjusted_rand_score(y, clusters)
silhouette = silhouette_score(X_all_scaled, clusters)
print(f"Adjusted Rand Index (ARI): {ari:.3f}  (1.0 = perfect match)")
print(f"Silhouette Score:          {silhouette:.3f}  (higher = better separation)")

# -- Dimensionality reduction with PCA for visualization ------
pca = PCA(n_components=2)
X_2d = pca.fit_transform(X_all_scaled)
print(f"PCA explained variance: {pca.explained_variance_ratio_.sum():.1%}")

# -- Visualization --------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Plot 1: K-Means clusters (no labels used)
ax1 = axes[0]
colors_cluster = ['#e74c3c', '#3498db', '#2ecc71']
for cluster_id in range(3):
    mask = clusters == cluster_id
    ax1.scatter(X_2d[mask, 0], X_2d[mask, 1], alpha=0.5,
                c=colors_cluster[cluster_id], label=f"Cluster {cluster_id}", s=30)
ax1.set_title("K-Means Clustering (unsupervised, no labels)", fontweight="bold")
ax1.set_xlabel("PCA Component 1")
ax1.set_ylabel("PCA Component 2")
ax1.legend()

# Plot 2: Real language labels (ground truth)
ax2 = axes[1]
colors_lang = {"pt": "#e74c3c", "en": "#3498db", "no": "#2ecc71"}
labels_lang = {"pt": "Portuguese", "en": "English", "no": "Norwegian"}
for lang in ["pt", "en", "no"]:
    mask = y == lang
    ax2.scatter(X_2d[mask, 0], X_2d[mask, 1], alpha=0.5,
                c=colors_lang[lang], label=labels_lang[lang], s=30)
ax2.set_title("Real Labels (ground truth)", fontweight="bold")
ax2.set_xlabel("PCA Component 1")
ax2.set_ylabel("PCA Component 2")
ax2.legend()

plt.suptitle("Do unsupervised clusters match real languages?", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print(f"\nARI = {ari:.3f} -- ", end="")
if ari > 0.7:
    print("Strong correspondence between clusters and languages!")
elif ari > 0.4:
    print("Moderate correspondence. Languages share some acoustic features.")
else:
    print("Weak correspondence. MFCC features alone may not fully separate languages.")

In [ ]:
#C30
!pip install -q jiwer

In [ ]:
## C33 - Ethical Reflection

### Privacy and voice data
- The local model version processes everything on the user's machine -- **no data leaves the device**
- The API version sends voice audio to Google's servers -- **privacy risk**
- Voice is biometric data: it can identify the speaker, their emotional state, accent, and origin
- In sensitive contexts (medical, legal), unauthorized voice processing raises serious ethical concerns

### Bias in AI models
- Whisper was trained predominantly on English data -- it recognizes English significantly better than Portuguese or Norwegian
- M2M100 may introduce cultural biases in translations (e.g., defaulting to masculine gender in gendered languages)
- SpeechT5 produces only one synthetic voice -- no diversity in vocal representation
- Under-resourced languages (like Norwegian) receive lower quality results, reinforcing digital inequality

### Accessibility
- The tool can help hearing-impaired individuals through real-time captions
- It can assist immigrants and refugees in daily communication
- However, it requires hardware (GPU) or internet (APIs), potentially excluding underserved populations

### Potential misuse
- Could be used for surveillance or monitoring conversations without consent
- Automatic translations in medical or legal contexts may cause harm if inaccurate
- Users might over-trust AI translations for critical communications

In [ ]:
## C34 - Model Quality and Limitations

### Observed quality
- **STT (Whisper)**: whisper-small achieves ~X% WER on Portuguese (FLEURS dataset)
  - Works best in quiet environments with clear speech
  - Struggles with strong regional accents and overlapping speakers
  - Performance degrades significantly for Norwegian compared to English
- **Translation (M2M100)**: Average BLEU score of ~X.XXX on PT->EN
  - Good for simple, direct sentences
  - Loses nuance with idiomatic expressions and cultural references
  - Norwegian translations show lower quality than English
- **TTS (SpeechT5)**: Robotic but intelligible output
  - Works best in English; Portuguese and Norwegian pronunciation is less natural

### Technical limitations
- Large models require GPU (not feasible on smartphones or low-end hardware)
- Total pipeline latency (~5-10s) prevents true real-time usage
- Whisper has a 30-second audio segment limit
- M2M100 418M sacrifices quality for speed; 1.2B is better but uses more VRAM
- Free Colab GPU sessions are limited to ~4 hours

### Possible improvements
- Fine-tune Whisper on PT-BR/Norwegian data to improve WER
- Use Voice Activity Detection (VAD) to record only when speech is detected
- Implement audio streaming to reduce latency
- Train custom speaker embeddings for more natural TTS voices
- Explore distillation techniques to create smaller, faster models